# 01_download：数据获取

本 Notebook 按作业要求完成以下任务：

1. 下载 10 只 A 股股票（后复权、日频），股票列表见 README.md
2. 下载指数：沪深300（必选）与中证500（自选）
3. 下载宏观指标：CPI 同比（必选）与 M2 同比（自选）
4. 下载 10 只股票最近 5 个年度财务指标（ROE、净利润率、资产负债率）并整理为长表
5. 记录统一下载日志到 `download_log.txt`

说明：为避免接口频率限制，下载过程加入了适度 `delay`。

## 任务准备：环境、路径、日志与通用函数

本节完成基础配置：

- 导入依赖库
- 定义项目路径与输出目录
- 初始化下载日志函数（统一 SUCCESS/FAILED 格式）
- 定义节流函数（避免请求过快）

In [26]:
from pathlib import Path
from datetime import datetime
import time
import random
import warnings

import pandas as pd
import akshare as ak

warnings.filterwarnings("ignore")

# 以当前 Notebook 所在项目目录作为根目录
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

# 确保目录存在
for sub in ["stock", "index", "macro", "finance"]:
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

LOG_PATH = ROOT / "download_log.txt"


def write_log(status: str, tag: str, message: str) -> None:
    """按作业要求记录下载日志。"""
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {status:<7} {tag:<15} {message}"
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(line)


def polite_delay(low: float = 0.6, high: float = 1.4) -> None:
    """随机延时，降低接口限频风险。"""
    time.sleep(random.uniform(low, high))


print(f"项目根目录: {ROOT}")
print(f"日志文件路径: {LOG_PATH}")

项目根目录: /Users/yijun/YJ/coding-workplace/ex_P02/dshw-p01
日志文件路径: /Users/yijun/YJ/coding-workplace/ex_P02/dshw-p01/download_log.txt


## 1.1 下载 10 只股票后复权日度行情

本节任务：

- 按既定股票池下载 2020-01-01 至今的后复权日线数据
- 保留并重命名关键字段：日期、开盘价、收盘价、最高价、最低价、成交量、成交额
- 每只股票保存为 `data/stock/stock_代码.csv`
- 每次下载结果写入 `download_log.txt`

说明：由于 `stock_zh_a_hist` 在部分网络环境下可能出现连接被远端关闭，本节改用 `stock_zh_a_daily(adjust='hfq')`，并加入重试机制以提升稳定性。

In [20]:
# 股票池：代码、名称、行业
stock_meta = [
    {"code": "600036", "name": "招商银行", "industry": "银行"},
    {"code": "601398", "name": "工商银行", "industry": "银行"},
    {"code": "002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "601238", "name": "广汽集团", "industry": "汽车"},
    {"code": "000002", "name": "万科A", "industry": "房地产"},
    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "601857", "name": "中国石油", "industry": "能源"},
    {"code": "600938", "name": "中国海油", "industry": "能源"},
    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},
    {"code": "002352", "name": "顺丰控股", "industry": "物流"},
]

start_date = "20200101"
end_date = datetime.now().strftime("%Y%m%d")

stock_frames = {}


def to_ak_symbol(code: str) -> str:
    # 上交所 6 开头使用 sh，深交所其余常见代码使用 sz
    return f"sh{code}" if code.startswith("6") else f"sz{code}"


for item in stock_meta:
    code = item["code"]
    tag = f"stock_{code}"
    symbol = to_ak_symbol(code)

    ok = False
    last_err = None

    # 连接不稳定时进行有限次重试，避免单次网络抖动导致整只股票失败
    for attempt in range(1, 4):
        try:
            df = ak.stock_zh_a_daily(
                symbol=symbol,
                start_date=start_date,
                end_date=end_date,
                adjust="hfq",
            )

            if df is None or df.empty:
                raise ValueError("No data returned")

            # stock_zh_a_daily 为英文列名，这里统一转换为作业要求中文字段
            required_cols = ["date", "open", "close", "high", "low", "volume", "amount"]
            miss = [c for c in required_cols if c not in df.columns]
            if miss:
                raise ValueError(f"Missing columns: {miss}")

            out = df[required_cols].copy().rename(
                columns={
                    "date": "日期",
                    "open": "开盘价",
                    "close": "收盘价",
                    "high": "最高价",
                    "low": "最低价",
                    "volume": "成交量",
                    "amount": "成交额",
                }
            )

            # 统一日期格式，避免后续清洗出现类型不一致
            out["日期"] = pd.to_datetime(out["日期"], errors="coerce").dt.strftime("%Y-%m-%d")
            out = out.dropna(subset=["日期"]).reset_index(drop=True)

            out["代码"] = code
            out["名称"] = item["name"]
            out["行业"] = item["industry"]

            save_path = DATA_DIR / "stock" / f"stock_{code}.csv"
            out.to_csv(save_path, index=False, encoding="utf-8-sig")

            stock_frames[code] = out
            write_log("SUCCESS", tag, f"shape={out.shape}")
            ok = True
            break
        except Exception as e:
            last_err = e
            # 重试间隔逐步增大，降低连续失败概率
            time.sleep(1.0 * attempt)

    if not ok:
        write_log("FAILED", tag, f"Error: {last_err}")

    polite_delay()

print(f"成功下载股票数: {len(stock_frames)}/{len(stock_meta)}")

[2026-04-07 20:27:55] SUCCESS stock_600036    shape=(1515, 10)
[2026-04-07 20:27:57] SUCCESS stock_601398    shape=(1515, 10)
[2026-04-07 20:27:59] SUCCESS stock_002594    shape=(1515, 10)
[2026-04-07 20:28:02] SUCCESS stock_601238    shape=(1515, 10)
[2026-04-07 20:28:04] SUCCESS stock_000002    shape=(1515, 10)
[2026-04-07 20:28:06] SUCCESS stock_600519    shape=(1515, 10)
[2026-04-07 20:28:08] SUCCESS stock_601857    shape=(1515, 10)
[2026-04-07 20:28:10] SUCCESS stock_600938    shape=(959, 10)
[2026-04-07 20:28:12] SUCCESS stock_000063    shape=(1514, 10)
[2026-04-07 20:28:14] SUCCESS stock_002352    shape=(1512, 10)
成功下载股票数: 10/10


## 1.2 下载市场指数：沪深300（必选）与中证500（自选）

本节任务：

- 下载沪深300（000300）与中证500（000905）日度数据
- 与股票使用一致时间区间（2020-01-01 至今）
- 保存为 `data/index/index_000300.csv`、`data/index/index_000905.csv`

选择中证500的理由：中证500覆盖中盘股，风格上与沪深300（大盘蓝筹）形成互补，可用于比较不同市值风格下的市场表现。

稳健策略：优先使用 `index_zh_a_hist`；若连接失败，则回退到 `stock_zh_index_daily` 并做字段标准化。

In [22]:
index_map = {
    "000300": "沪深300",
    "000905": "中证500",
}


def to_index_symbol(code: str) -> str:
    # 这两个指数均按上交所指数代码格式访问
    return f"sh{code}"


index_frames = {}
for code, name in index_map.items():
    tag = f"index_{code}"
    ok = False
    last_err = None

    # 最多重试 3 次
    for attempt in range(1, 4):
        try:
            # 主接口：A 股指数历史（部分网络环境可能不稳定）
            df = ak.index_zh_a_hist(
                symbol=code,
                period="daily",
                start_date=start_date,
                end_date=end_date,
            )

            if df is None or df.empty:
                raise ValueError("No data returned from primary API")

            keep = [c for c in ["日期", "开盘", "收盘", "最高", "最低", "成交量", "成交额"] if c in df.columns]
            out = df[keep].copy().rename(
                columns={
                    "开盘": "开盘价",
                    "收盘": "收盘价",
                    "最高": "最高价",
                    "最低": "最低价",
                }
            )
            out["代码"] = code
            out["名称"] = name

            save_path = DATA_DIR / "index" / f"index_{code}.csv"
            out.to_csv(save_path, index=False, encoding="utf-8-sig")

            index_frames[code] = out
            write_log("SUCCESS", tag, f"shape={out.shape} source=index_zh_a_hist")
            ok = True
            break
        except Exception:
            # 备用接口：指数日线（字段为英文，需标准化）
            try:
                df2 = ak.stock_zh_index_daily(symbol=to_index_symbol(code))
                if df2 is None or df2.empty:
                    raise ValueError("No data returned from fallback API")

                # 兼容两种结构：date 在列中 or date 在索引中
                if "date" in df2.columns:
                    tmp = df2.copy()
                else:
                    tmp = df2.reset_index().rename(columns={"index": "date"}).copy()

                tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
                tmp = tmp.dropna(subset=["date"])

                # 按目标区间过滤
                s_dt = pd.to_datetime(start_date, format="%Y%m%d", errors="coerce")
                e_dt = pd.to_datetime(end_date, format="%Y%m%d", errors="coerce")
                tmp = tmp[(tmp["date"] >= s_dt) & (tmp["date"] <= e_dt)]

                # 统一字段；若无 amount 列则以空值占位
                if "amount" not in tmp.columns:
                    tmp["amount"] = pd.NA

                out = tmp[["date", "open", "close", "high", "low", "volume", "amount"]].rename(
                    columns={
                        "date": "日期",
                        "open": "开盘价",
                        "close": "收盘价",
                        "high": "最高价",
                        "low": "最低价",
                        "volume": "成交量",
                        "amount": "成交额",
                    }
                )
                out["日期"] = out["日期"].dt.strftime("%Y-%m-%d")
                out["代码"] = code
                out["名称"] = name

                save_path = DATA_DIR / "index" / f"index_{code}.csv"
                out.to_csv(save_path, index=False, encoding="utf-8-sig")

                index_frames[code] = out
                write_log("SUCCESS", tag, f"shape={out.shape} source=stock_zh_index_daily")
                ok = True
                break
            except Exception as e2:
                last_err = e2
                time.sleep(1.0 * attempt)

    if not ok:
        write_log("FAILED", tag, f"Error: {last_err}")

    polite_delay()

print(f"成功下载指数数: {len(index_frames)}/{len(index_map)}")

[2026-04-07 20:29:09] SUCCESS index_000300    shape=(1515, 9) source=stock_zh_index_daily
[2026-04-07 20:29:10] SUCCESS index_000905    shape=(1515, 9) source=stock_zh_index_daily
成功下载指数数: 2/2


## 1.3 下载宏观指标：CPI 同比（必选）与 M2 同比（自选）

本节任务：

- 下载 CPI 同比增速（月度）
- 下载 M2 同比增速（月度）
- 分别保存为 `data/macro/macro_cpi.csv`、`data/macro/macro_m2_yoy.csv`

选择 M2 同比的理由：M2 同比反映货币环境松紧，通常影响市场流动性与风险偏好，对股票估值与风格切换具有解释意义。

In [23]:
# CPI 同比
try:
    cpi_df = ak.macro_china_cpi_yearly()
    if cpi_df is None or cpi_df.empty:
        raise ValueError("No data returned")

    cpi_out = cpi_df.copy()
    cpi_out.to_csv(DATA_DIR / "macro" / "macro_cpi.csv", index=False, encoding="utf-8-sig")
    write_log("SUCCESS", "macro_cpi", f"shape={cpi_out.shape}")
except Exception as e:
    cpi_out = pd.DataFrame()
    write_log("FAILED", "macro_cpi", f"Error: {e}")

polite_delay()

# M2 同比
try:
    m2_df = ak.macro_china_money_supply()
    if m2_df is None or m2_df.empty:
        raise ValueError("No data returned")

    # 在不同版本接口下，列名可能不同，这里兼容常见写法
    value_col_candidates = ["同比", "m2同比", "M2同比", "m2_yoy", "M2_yoy"]
    value_col = next((c for c in value_col_candidates if c in m2_df.columns), None)

    # 若没有匹配列，直接保留原表并由后续清洗阶段统一处理
    m2_out = m2_df.copy()
    if value_col is not None:
        m2_out = m2_out.rename(columns={value_col: "m2同比"})

    m2_out.to_csv(DATA_DIR / "macro" / "macro_m2_yoy.csv", index=False, encoding="utf-8-sig")
    write_log("SUCCESS", "macro_m2", f"shape={m2_out.shape}")
except Exception as e:
    m2_out = pd.DataFrame()
    write_log("FAILED", "macro_m2", f"Error: {e}")

polite_delay()

[2026-04-07 20:29:18] SUCCESS macro_cpi       shape=(477, 5)
[2026-04-07 20:29:20] SUCCESS macro_m2        shape=(218, 10)


## 1.4 下载财务指标并整理为长格式（Long Format）

本节任务：

- 针对 10 只股票获取最近 5 个年度财务指标
- 指标包含：ROE、净利润率、资产负债率
- 统一整理为长表，字段为：`code, year, indicator, value`
- 保存为 `data/finance/finance_ratios.csv`

稳健策略：优先使用 `stock_financial_analysis_indicator`；若返回空表或连接异常，则回退到 `stock_financial_abstract`，从“指标 × 报告期列”的结构中抽取三项指标并转换为长表。

In [27]:
current_year = datetime.now().year
# 使用最近5个"完整"年度，避免把未披露完整的当年财报算入窗口
end_fin_year = current_year - 1
last_5_years = {str(y) for y in range(end_fin_year - 4, end_fin_year + 1)}

# 财务指标列名的兼容匹配关键词（主接口）
indicator_map = {
    "roe": ["净资产收益率", "ROE"],
    "net_profit_margin": ["销售净利率", "净利率", "净利润率"],
    "debt_asset_ratio": ["资产负债率"],
}

# 财务摘要接口中“指标”列的关键词（备用接口）
abstract_indicator_map = {
    "roe": ["净资产收益率(ROE)", "净资产收益率"],
    "net_profit_margin": ["销售净利率", "净利率", "净利润率"],
    "debt_asset_ratio": ["资产负债率"],
}


def find_col_by_keywords(columns, keywords):
    for col in columns:
        if any(k in str(col) for k in keywords):
            return col
    return None


def extract_from_analysis_api(code: str) -> pd.DataFrame:
    """主接口：stock_financial_analysis_indicator，返回长表。"""
    fin = ak.stock_financial_analysis_indicator(symbol=code)
    if fin is None or fin.empty:
        raise ValueError("No data returned from primary API")

    date_col = "日期" if "日期" in fin.columns else fin.columns[0]
    fin[date_col] = pd.to_datetime(fin[date_col], errors="coerce")
    fin = fin.dropna(subset=[date_col]).copy()
    fin["year"] = fin[date_col].dt.year.astype(str)

    fin_year = fin.sort_values(date_col).groupby("year", as_index=False).tail(1)
    fin_year = fin_year[fin_year["year"].isin(last_5_years)]
    if fin_year.empty:
        raise ValueError("No records in last 5 years from primary API")

    records = []
    for indicator, keywords in indicator_map.items():
        col = find_col_by_keywords(fin_year.columns, keywords)
        if col is None:
            continue
        tmp = fin_year[["year", col]].copy()
        tmp["value"] = pd.to_numeric(tmp[col], errors="coerce")
        tmp = tmp.dropna(subset=["value"])
        tmp["code"] = code
        tmp["indicator"] = indicator
        records.extend(tmp[["code", "year", "indicator", "value"]].to_dict("records"))

    out = pd.DataFrame(records)
    if out.empty:
        raise ValueError("No target indicators extracted from primary API")
    return out


def extract_from_abstract_api(code: str) -> pd.DataFrame:
    """备用接口：stock_financial_abstract（指标为行，报告期为列）。"""
    abs_df = ak.stock_financial_abstract(symbol=code)
    if abs_df is None or abs_df.empty:
        raise ValueError("No data returned from fallback API")

    if "指标" not in abs_df.columns:
        raise ValueError("Fallback API missing 指标 column")

    # 报告期列形如 20241231，仅保留最近 5 年年报(1231)
    date_cols = [
        c for c in abs_df.columns
        if isinstance(c, str) and len(c) == 8 and c.isdigit() and c.endswith("1231") and c[:4] in last_5_years
    ]
    if not date_cols:
        raise ValueError("No annual report columns found in fallback API")

    date_cols = sorted(date_cols)

    records = []
    for indicator, keywords in abstract_indicator_map.items():
        mask = abs_df["指标"].astype(str).apply(lambda x: any(k in x for k in keywords))
        cand = abs_df.loc[mask].copy()
        if cand.empty:
            continue

        # 若匹配到多行，优先取第一条最核心口径
        row = cand.iloc[0]
        for dc in date_cols:
            val = pd.to_numeric(row.get(dc), errors="coerce")
            if pd.isna(val):
                continue
            records.append(
                {
                    "code": code,
                    "year": dc[:4],
                    "indicator": indicator,
                    "value": float(val),
                }
            )

    out = pd.DataFrame(records)
    if out.empty:
        raise ValueError("No target indicators extracted from fallback API")

    # 每年每指标去重（若有重复口径）
    out = out.sort_values(["year", "indicator"]).drop_duplicates(["code", "year", "indicator"], keep="first")
    return out


finance_long_records = []

for item in stock_meta:
    code = item["code"]
    tag = f"finance_{code}"
    ok = False
    last_err = None

    for attempt in range(1, 4):
        try:
            out = extract_from_analysis_api(code)
            finance_long_records.extend(out.to_dict("records"))
            write_log("SUCCESS", tag, f"shape={out.shape} source=analysis_indicator")
            ok = True
            break
        except Exception as e1:
            last_err = e1
            try:
                out = extract_from_abstract_api(code)
                finance_long_records.extend(out.to_dict("records"))
                write_log("SUCCESS", tag, f"shape={out.shape} source=financial_abstract")
                ok = True
                break
            except Exception as e2:
                last_err = e2
                time.sleep(1.0 * attempt)

    if not ok:
        write_log("FAILED", tag, f"Error: {last_err}")

    polite_delay()

finance_long = pd.DataFrame(finance_long_records)
if not finance_long.empty:
    finance_long = (
        finance_long.sort_values(["code", "year", "indicator"])
        .drop_duplicates(["code", "year", "indicator"], keep="first")
        .reset_index(drop=True)
    )

finance_save_path = DATA_DIR / "finance" / "finance_ratios.csv"
finance_long.to_csv(finance_save_path, index=False, encoding="utf-8-sig")

print(f"finance_long shape: {finance_long.shape}")
finance_long.head(12)

  0%|          | 0/28 [00:00<?, ?it/s]

[2026-04-08 13:53:50] SUCCESS finance_600036  shape=(15, 4) source=analysis_indicator
[2026-04-08 13:53:52] SUCCESS finance_601398  shape=(15, 4) source=financial_abstract
[2026-04-08 13:53:54] SUCCESS finance_002594  shape=(15, 4) source=financial_abstract
[2026-04-08 13:53:55] SUCCESS finance_601238  shape=(15, 4) source=financial_abstract
[2026-04-08 13:53:57] SUCCESS finance_000002  shape=(15, 4) source=financial_abstract
[2026-04-08 13:54:00] SUCCESS finance_600519  shape=(12, 4) source=financial_abstract
[2026-04-08 13:54:02] SUCCESS finance_601857  shape=(15, 4) source=financial_abstract
[2026-04-08 13:54:04] SUCCESS finance_600938  shape=(15, 4) source=financial_abstract
[2026-04-08 13:54:05] SUCCESS finance_000063  shape=(15, 4) source=financial_abstract
[2026-04-08 13:54:07] SUCCESS finance_002352  shape=(15, 4) source=financial_abstract
finance_long shape: (147, 4)


,code,year,indicator,value
0,000002,2021,debt_asset_ratio,79.739757
1,000002,2021,net_profit_margin,8.407622
2,000002,2021,roe,9.780000
3,000002,2022,debt_asset_ratio,76.951461
4,000002,2022,net_profit_margin,7.452967
5,000002,2022,roe,9.480000
6,000002,2023,debt_asset_ratio,73.224342
7,000002,2023,net_profit_margin,4.392064
8,000002,2023,roe,4.910000
9,000002,2024,debt_asset_ratio,73.655816


## 1.5 下载日志与结果抽样检查

本节任务：

- 预览 `download_log.txt` 最近日志，核对 SUCCESS/FAILED 记录格式
- 抽样查看已下载文件是否存在，便于快速确认任务完成情况

In [25]:
# 预览日志末尾 20 行
if LOG_PATH.exists():
    print("===== download_log.txt (tail 20) =====")
    lines = LOG_PATH.read_text(encoding="utf-8").splitlines()
    for line in lines[-20:]:
        print(line)
else:
    print("日志文件不存在。")

print("\n===== 文件存在性检查 =====")
check_paths = [
    DATA_DIR / "stock" / "stock_600036.csv",
    DATA_DIR / "index" / "index_000300.csv",
    DATA_DIR / "index" / "index_000905.csv",
    DATA_DIR / "macro" / "macro_cpi.csv",
    DATA_DIR / "macro" / "macro_m2_yoy.csv",
    DATA_DIR / "finance" / "finance_ratios.csv",
]
for p in check_paths:
    print(f"{p.name:<25} -> {'OK' if p.exists() else 'MISSING'}")

===== download_log.txt (tail 20) =====
[2026-04-07 20:28:08] SUCCESS stock_601857    shape=(1515, 10)
[2026-04-07 20:28:10] SUCCESS stock_600938    shape=(959, 10)
[2026-04-07 20:28:12] SUCCESS stock_000063    shape=(1514, 10)
[2026-04-07 20:28:14] SUCCESS stock_002352    shape=(1512, 10)
[2026-04-07 20:28:30] FAILED  index_000300    Error: cannot assemble with duplicate keys
[2026-04-07 20:28:38] FAILED  index_000905    Error: cannot assemble with duplicate keys
[2026-04-07 20:29:09] SUCCESS index_000300    shape=(1515, 9) source=stock_zh_index_daily
[2026-04-07 20:29:10] SUCCESS index_000905    shape=(1515, 9) source=stock_zh_index_daily
[2026-04-07 20:29:18] SUCCESS macro_cpi       shape=(477, 5)
[2026-04-07 20:29:20] SUCCESS macro_m2        shape=(218, 10)
[2026-04-07 20:29:30] SUCCESS finance_600036  shape=(12, 4) source=analysis_indicator
[2026-04-07 20:29:32] SUCCESS finance_601398  shape=(12, 4) source=financial_abstract
[2026-04-07 20:29:34] SUCCESS finance_002594  shape=(12, 